# 07.1 — Temporal rescoring and WBF on top of the 07 pipeline
Same model selection as `07_pipeline_variants`, with two additions: Weighted Boxes Fusion (`*_wbf*` variants and `ens_top*_wbf` next to the NMS ones) and temporal rescoring of the winner's scores over neighbouring frames of the same clip (`vod.temporal.rescore`).

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/vod").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q ultralytics faster-coco-eval

In [ ]:
import ast
import shutil

import pandas as pd
import torch

from vod import CLASSES
from vod.data import find_data_root, load_split
from vod.evaluate import coco_evaluate, per_clip_map
from vod.protocol import PROTOCOL
from vod.submission import write_submission
from vod.tv import comparison_tables, completed_runs, load_run_model
from vod.variants import (choose_variant, collect_view, compare_variants, merge_views, torchvision_detector,
                          yolo_detector)

DATA = find_data_root()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
ROOTS = (WORK, Path("/kaggle/input"))

## Models and variants

In [ ]:
VAL_STEP = 5
LOW_RES = 960  # test has 680 and 960 px clips, val starts at 1344
BATCH = 4
REF = "full"
VARIANTS = {
    REF: dict(views=["full"]),
    "flip_nms50": dict(views=["full", "flip"], iou=0.5),
    "flip_nms65": dict(views=["full", "flip"], iou=0.65),
    "tiles_nms50": dict(views=["full", "tiles"], iou=0.5),
    "tiles_nms65": dict(views=["full", "tiles"], iou=0.65),
    "flip_tiles_nms50": dict(views=["full", "flip", "tiles"], iou=0.5),
    "flip_tiles_nms65": dict(views=["full", "flip", "tiles"], iou=0.65),
}
VARIANTS.update({k.replace("nms", "wbf"): {**v, "fuse": "wbf"} for k, v in list(VARIANTS.items())
                 if len(v["views"]) > 1})  # the same views and IoU, fused by WBF instead of NMS
VIEWS = sorted({n for v in VARIANTS.values() for n in v["views"]})

In [ ]:
def has_weights(pattern):
    return any(root.exists() and any(root.rglob(pattern)) for root in ROOTS)


def make_detector(name):
    if name.startswith(("yolo", "rtdetr")):
        from vod.yolo import load_ultralytics
        pattern = "baseline/weights/best.pt" if name == "yolo_baseline" else f"{name}/weights/last.pt"
        weights = next(p for root in ROOTS if root.exists() for p in root.rglob(pattern))
        return yolo_detector(load_ultralytics(weights))
    cfg = runs[name]["config"]
    return torchvision_detector(load_run_model(name, cfg, ROOTS), cfg)


runs = completed_runs(ROOTS)
candidates = ["yolo_baseline"] if has_weights("baseline/weights/best.pt") else []
if runs:
    main = comparison_tables(runs, CLASSES)[0]
    for prefix, pattern in [("yolo_", "{}/weights/last.pt"), ("rtdetr", "{}/weights/last.pt"),
                            ("frcnn", "{}/last.pt"), ("retina", "{}/last.pt"), ("fcos", "{}/last.pt")]:
        family = [n for n in main[main.index.str.startswith(prefix)].sort_values("mAP", ascending=False).index
                  if has_weights(pattern.format(n))]
        candidates += family[:1]
print("candidates:", candidates)

## Comparison on val: native resolution and shrunk to the low-res test clips

In [ ]:
val_frames, _ = load_split(DATA / "val", step=1)
VAL_GT = DATA / "val" / "annotations_coco.json"
paths = val_frames.iloc[::VAL_STEP]["path"].tolist()
print(len(paths), "val frames")

results = {}
for name in candidates:
    detect = make_detector(name)
    views_native = {v: collect_view(detect, paths, v, batch=BATCH) for v in VIEWS}
    views_lowres = {v: collect_view(detect, paths, v, max_side=LOW_RES, batch=BATCH) for v in VIEWS}
    native = compare_variants(views_native, VARIANTS, VAL_GT, paths, REF)
    lowres = compare_variants(views_lowres, VARIANTS, VAL_GT, paths, REF, clips=False)
    chosen = choose_variant(native, lowres, REF)
    pick = VARIANTS[chosen]
    results[name] = dict(native=native, lowres=lowres, chosen=chosen,
                         dets_native=merge_views([views_native[v] for v in pick["views"]], pick.get("iou"), pick.get("fuse", "nms")),
                         dets_lowres=merge_views([views_lowres[v] for v in pick["views"]], pick.get("iou"), pick.get("fuse", "nms")))
    native.to_csv(WORK / f"variants_{name}_native.csv")
    lowres.to_csv(WORK / f"variants_{name}_lowres.csv")
    print(name, "| chosen:", chosen)
    display(native)
    display(lowres)
    del detect
    torch.cuda.empty_cache()

In [ ]:
summary = pd.DataFrame({
    n: {"chosen": r["chosen"],
        "mAP_ref": r["native"].loc[REF, "mAP"], "mAP": r["native"].loc[r["chosen"], "mAP"],
        "mAP_lowres_ref": r["lowres"].loc[REF, "mAP"], "mAP_lowres": r["lowres"].loc[r["chosen"], "mAP"],
        "clips_better": r["native"].loc[r["chosen"], "clips_better"],
        "ms_per_frame": r["native"].loc[r["chosen"], "ms_per_frame"]}
    for n, r in results.items()}).T
summary["score"] = (summary["mAP"] + summary["mAP_lowres"]) / 2
summary = summary.sort_values("score", ascending=False)
summary

## Ensembles of the best models
Must beat the best single model at both resolutions and in most clips.

In [ ]:
ENS_IOU = 0.5
best = summary.index[0]
best_clips = pd.Series(per_clip_map(VAL_GT, results[best]["dets_native"], paths))
ensembles = {f"ens_top{k}{s}": (list(summary.index[:k]), f) for k in range(2, len(summary) + 1)
             for s, f in [("", "nms"), ("_wbf", "wbf")]}
rows = {}
for ens, (members, fuse) in ensembles.items():
    native = merge_views([results[n]["dets_native"] for n in members], ENS_IOU, fuse=fuse)
    lowres = merge_views([results[n]["dets_lowres"] for n in members], ENS_IOU, fuse=fuse)
    clips = pd.Series(per_clip_map(VAL_GT, native, paths))
    rows[ens] = {"mAP": coco_evaluate(VAL_GT, native, file_names=paths)["mAP"],
                 "mAP_lowres": coco_evaluate(VAL_GT, lowres, file_names=paths)["mAP"],
                 "clips_better": int((clips > best_clips).sum())}
ens_table = pd.DataFrame(rows).T
if len(ens_table):
    ens_table["score"] = (ens_table["mAP"] + ens_table["mAP_lowres"]) / 2
    display(ens_table.round(4))
pd.concat([summary, ens_table]).to_csv(WORK / "variants_summary.csv")

## Winner

In [ ]:
WINNER, ENS_FUSE = [best], "nms"
if len(ens_table):
    ok = ens_table[(ens_table["mAP"] > summary.loc[best, "mAP"])
                   & (ens_table["mAP_lowres"] > summary.loc[best, "mAP_lowres"])
                   & (ens_table["clips_better"] > len(best_clips) / 2)]
    if len(ok):
        WINNER, ENS_FUSE = ensembles[ok["score"].idxmax()]
print("winner:", {n: results[n]["chosen"] for n in WINNER}, "| ensemble fusion:", ENS_FUSE)


def detections(members, paths, cache):
    """Detections of the members, each in its chosen variant, merged; `cache` keeps the views per (model, view)."""
    parts = []
    for name in members:
        pick = VARIANTS[results[name]["chosen"]]
        missing = [v for v in pick["views"] if (name, v) not in cache]
        if missing:
            detect = make_detector(name)
            for v in missing:
                cache[name, v] = collect_view(detect, paths, v, batch=BATCH)
            del detect
            torch.cuda.empty_cache()
        parts.append(merge_views([cache[name, v] for v in pick["views"]], pick.get("iou"), pick.get("fuse", "nms")))
    return merge_views(parts, ENS_IOU, fuse=ENS_FUSE) if len(parts) > 1 else parts[0]

## Temporal rescoring on val
A box that also appears in the neighbouring frames of its clip keeps its score, a box seen in one frame loses part of it. The test has every frame of each clip, so val is scored on consecutive frames too (`TEMP_STEP = 1`); with 5 the neighbours are 5 frames apart and the links are weaker.

In [ ]:
from vod.temporal import rescore

TEMP_STEP = 1
val_t = val_frames.iloc[::TEMP_STEP]["path"].tolist()
dets_val = detections(WINNER, val_t, {})
base = coco_evaluate(VAL_GT, dets_val, file_names=val_t)["mAP"]

rows = {}
for window in (1, 2):
    for alpha in (0.3, 0.6):
        for iou in (0.5, 0.7):
            rescored = rescore(dets_val, val_t, window=window, iou=iou, alpha=alpha)
            rows[window, alpha, iou] = coco_evaluate(VAL_GT, rescored, file_names=val_t)["mAP"]
sweep = pd.Series(rows, name="mAP").rename_axis(["window", "alpha", "iou"]).to_frame()
sweep["gain"] = sweep["mAP"] - base
display(sweep.round(4))
print("without rescoring:", round(base, 4))

In [ ]:
window, alpha, iou = sweep["mAP"].idxmax()
clips_base = pd.Series(per_clip_map(VAL_GT, dets_val, val_t))
clips_temp = pd.Series(per_clip_map(VAL_GT, rescore(dets_val, val_t, window=window, iou=iou, alpha=alpha), val_t))
# same rule as for the variants: it has to help in most clips, not in one
USE_TEMPORAL = sweep["mAP"].max() > base and (clips_temp > clips_base).sum() > len(clips_base) / 2
print(dict(window=window, alpha=alpha, iou=iou), "| clips better:", int((clips_temp > clips_base).sum()), "of", len(clips_base),
      "| use:", USE_TEMPORAL)
pd.DataFrame({"without": clips_base, "rescored": clips_temp})

## Test predictions
Both files are written, so the leaderboard can say whether the val gain holds.

In [ ]:
test = pd.read_csv(DATA / "test_images.csv")
test_paths = [str(DATA / f) for f in test["file_name"]]
clip_sizes = pd.Series([Path(p).parent.name for p in test_paths]).value_counts()
print(len(clip_sizes), "clips, frames per clip:", clip_sizes.min(), "-", clip_sizes.max())  # links must not run across clips
dets_test = detections(WINNER, test_paths, {})


def write(dets, suffix):
    out = dets.copy()
    out["image_id"] = out.pop("file_name").map(dict(zip(test_paths, test["image_id"])))
    return write_submission(out, DATA / "test_images.csv", WORK / f"submission_{'+'.join(WINNER)}_{suffix}.csv")


write(dets_test, "plain")
if USE_TEMPORAL:
    sub = write(rescore(dets_test, test_paths, window=window, iou=iou, alpha=alpha), "temporal")
    display(sub.head())